In [20]:
import pandas as pd

In [21]:
# Time-based revenue pattern
# medium
# The store owner suspects weekday mornings are slowest. The Time column contains values like "10:30" and Date contains "1/5/2019".

# Parse both columns into proper datetime types. Extract the hour of day from Time and the day of week from Date.
#Then find: which hour has the highest total revenue? Which day of week has the lowest?

# Print a clean summary table showing revenue by hour, sorted descending.
df = pd.read_csv('SuperMarket Analysis.csv')


In [22]:
#Date column ko text se real Date format mein badlein
df['Date'] = pd.to_datetime(df['Date'],format='%m/%d/%Y')

# Time column ko text se real Time format mein badlein



# Hum iske aage .dt.time laga rahe the, jiski wajah se Pandas is column ka "Datetime status" khatam kar ke
# use core Python ke time object mein badal raha tha. Phir agli line par .dt kam karna chhor deta tha.
df['Time'] = pd.to_datetime(df['Time'], format='mixed')



# Ab kya ho gaya: Time column abhi bhi Pandas ke under control hai
# (is ka Datetime status active hai). Is wajah se iska
# .dt wala darwaza khula raha, taake agli line is par apna kaam kar sake.

In [23]:
# Time mein se ghanta nikalna (e.g., 13, 14, 15...)
# Kyunki humne upar use pure time object banaya tha, use pehle string se hour mein convert kar lete hain



# Back-end par kya hua: Agar time 13:08:00 (yani 1:08 PM) tha, toh Pandas ne baaki sab chhor 
# kar sirf 13 nikala aur use naye Hour column mein save kar diya.



# Pehle jab hum dobara pd.to_datetime() laga rahe the, toh Pandas kehta tha
# ke "Bhai pehle hi convert kar chuke ho, 
# ab dobara kyun text samajh rahe ho?" aur crash ho jata tha.


# Correct approach — extract hour directly from string
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour
# No need to store the full datetime — just the hour you need



# .day_name() ek built-in function hai. Yeh khud-ba-khud piche calendar
# check karta hai ke us date ko kya din tha—Monday tha, Tuesday tha, ya Saturday


# Usne har date ka asli din dhoond kar (jaise Monday, Tuesday, Saturday) 
# ek naya column Day_of_Week bana diya.

df['Day_of_Week'] = df['Date'].dt.day_name()



In [24]:
hourly_revenue = df.groupby('Hour')['Sales'].sum().sort_values(ascending=False)
print("--- REVENUE BY HOUR (Highest to Lowest) ---")
print(hourly_revenue)

print("\n" + "="*40 + "\n")

# 4. Kis din sabse kam sales hui (Lowest to Highest)
daily_revenue = df.groupby('Day_of_Week')['Sales'].sum().sort_values(ascending=True)
print("--- REVENUE BY DAY OF WEEK (Lowest to Highest) ---")
print(daily_revenue)




# peak_hour = hourly_revenue.idxmax()
# slowest_day = daily_revenue.idxmin()
# print(f"Peak sales hour: {peak_hour}:00 with revenue {hourly_revenue.max():,.0f}")
# print(f"Slowest day: {slowest_day} with revenue {daily_revenue.min():,.0f}")
# print(f"The owner's hypothesis about slow mornings is: {'CORRECT' if hourly_revenue[10] < hourly_revenue.median() else 'INCORRECT'}")

--- REVENUE BY HOUR (Highest to Lowest) ---
Hour
19    39699.5130
13    34723.2270
10    31421.4810
15    31179.5085
14    30828.3990
11    30377.3295
12    26065.8825
18    26030.3400
16    25226.3235
17    24445.2180
20    22969.5270
Name: Sales, dtype: float64


--- REVENUE BY DAY OF WEEK (Lowest to Highest) ---
Day_of_Week
Monday       37899.0780
Wednesday    43731.1350
Friday       43926.3405
Sunday       44457.8925
Thursday     45349.2480
Tuesday      51482.2455
Saturday     56120.8095
Name: Sales, dtype: float64


In [25]:
# The owner wants to understand who his best customers are. Using Gender, Payment, and Customer type columns together:

# 1. Create a pivot table: rows = Customer type (Member vs Normal), columns = Payment method, values = average Total spend.
# 2. Which combination (e.g. "Member + Ewallet") spends the most on average?
# 3. Calculate what percentage of total revenue comes from Member customers vs Normal customers.
#Format the output as: "Members contribute X.X% of revenue"

In [26]:

# cust_type index  = mean  row hai or yeh col row mn dikhae ga neeche
# payment   col o=to yeh col bn jae ga  
# Sales ka mean normal k liye alag cash category nikly ga, credit card k liye alag, or ewallet k liye alag
# Sales ka mean Member k liye alag cash category nikly ga, credit card k liye alag, or ewallet k liye alag

customer_pivot = df.pivot_table(index='Customer type', 
                                columns='Payment', 
                                values='Sales', 
                                aggfunc='mean')
print("--- CUSTOMER SPENDING PIVOT TABLE (AVERAGE) ---")
print(customer_pivot)

--- CUSTOMER SPENDING PIVOT TABLE (AVERAGE) ---
Payment              Cash  Credit card     Ewallet
Customer type                                     
Member         335.034602   339.734406  332.461218
Normal         314.999516   300.296274  302.863651


In [27]:
# Pata lagane ke liye ke kis combination ka average sabse zyada hai

# Toh pehle .max() ne humein poori table mein se sirf ek number nahi diya, balki teeno columns ke bade numbers ki ek list de di:
# Cash: 335.03
# Credit card: 339.73
# Ewallet: 332.46

# Dusri dafa ka .max(): Us List mein se sabse bada number nikalta hai

highest_avg = customer_pivot.max().max()
print(f"\nHighest average spend in the table is: {highest_avg:.2f}")





# Find the winner programmatically, don't make the client scan the table
# best_combo = customer_pivot.stack().idxmax()
# best_value = customer_pivot.stack().max()
# print(f"Highest spending combination: {best_combo[0]} + {best_combo[1]} → Avg spend: {best_value:.2f}")


Highest average spend in the table is: 339.73


In [28]:
# 1. Pooray store ki kul jama (total) sales
total_store_revenue = df['Sales'].sum()

# 2. Har customer type ki alag alag total sales nikalna
revenue_by_type = df.groupby('Customer type')['Sales'].sum()

# 3. Members ki percentage calculate karna
member_revenue = revenue_by_type['Member']
member_percentage = (member_revenue / total_store_revenue) * 100

# 4. Final output ko owner ke bataye hue format mein print karna
print(f"Members contribute {member_percentage:.1f}% of revenue")

Members contribute 58.7% of revenue


In [29]:
df

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,Hour,Day_of_Week
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-01-05,2026-06-21 13:08:00,Ewallet,522.83,4.761905,26.1415,9.1,13,Saturday
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,2019-03-08,2026-06-21 10:29:00,Cash,76.40,4.761905,3.8200,9.6,10,Friday
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03,2026-06-21 13:23:00,Credit card,324.31,4.761905,16.2155,7.4,13,Sunday
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,2019-01-27,2026-06-21 20:33:00,Ewallet,465.76,4.761905,23.2880,8.4,20,Sunday
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2019-02-08,2026-06-21 10:37:00,Ewallet,604.17,4.761905,30.2085,5.3,10,Friday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,233-67-5758,Giza,Naypyitaw,Normal,Male,Health and beauty,40.35,1,2.0175,42.3675,2019-01-29,2026-06-21 13:46:00,Ewallet,40.35,4.761905,2.0175,6.2,13,Tuesday
996,303-96-2227,Cairo,Mandalay,Normal,Female,Home and lifestyle,97.38,10,48.6900,1022.4900,2019-03-02,2026-06-21 17:16:00,Ewallet,973.80,4.761905,48.6900,4.4,17,Saturday
997,727-02-1313,Alex,Yangon,Member,Male,Food and beverages,31.84,1,1.5920,33.4320,2019-02-09,2026-06-21 13:22:00,Cash,31.84,4.761905,1.5920,7.7,13,Saturday
998,347-56-2442,Alex,Yangon,Normal,Male,Home and lifestyle,65.82,1,3.2910,69.1110,2019-02-22,2026-06-21 15:33:00,Cash,65.82,4.761905,3.2910,4.1,15,Friday


In [30]:
# The owner has 3 branches (A, B, C). Build a branch scorecard — a single DataFrame that shows, for each branch:

# • Total revenue  •  Average transaction value  •  Average customer rating  •  Number of transactions  •  Most popular product line

# Then add a rank column that ranks branches 1–3 by total revenue.

# Finally write 2–3 sentences in a Markdown cell: "Based on this data, I recommend the owner focus on Branch X because...
# " — this is your first real client recommendation.




In [31]:
# Isme hum groupby ke saath .agg() (Aggregation) ka use karenge taake ek hi dafa mein saari calculations ho jayein.

import pandas as pd

# 1. Har branch ke liye saari required cheezein calculate karna
# Jab bhi sawal mein yeh alfaaz aayein: "For each...", "Per...", "Group-wise..." (jaise har branch ke liye, har product ke liye, har city ke liye),
# toh iska 100% matlab yeh hota hai ke wahan groupby lagega.


# Pehle poore data ko Branch ke mutabiq 3 alag-alag dheron (groups) mein baanto—Branch
# A ka dher alag, B ka alag, aur C ka alag. Phir har dher ke upar alag se calculations karo.


# .groupby('Branch'): Pandas ko bola ke data ko 'Branch' (A, B, C) ke naam par 3 hisson mein divide kar do.
branch_scorecard = df.groupby('Branch').agg(
    Total_Revenue=('Sales', 'sum'),
    Avg_Transaction_Value=('Sales', 'mean'),
    Avg_Rating=('Rating', 'mean'),
    Num_Transactions=('Invoice ID', 'count'),
    
    # Math mein Mode ka matlab hota hai "woh cheez jo sabse zyada baar aaye". [0] 
# lagane ka matlab hai ke agar do cheezein barabar hon, toh pehli wali utha lo
    
    Most_Popular_Product=('Product line', lambda x: x.mode()[0])
).reset_index()

# 2. Total Revenue ke mutabiq Ranking dena (Highest revenue = Rank 1)



# Hum Pandas ko keh rahe hain ke hamari scorecard wali table ke andar ek naya column bana do aur uska naam rakh do Rank
# Pandas ko bata rahe hain ke ranking kis cheez ki buniyaad (basis) par karni hai. Humne kaha: "Bhai, Total_Revenue


# Yeh settings tab kaam aati hai jab Tie (Match Draw) ho jaye. Misaal ke taur par, agar Branch A aur Branch B 
# dono ne bilkul barabar $50,000 kamaye hon, toh dono ko kaun sa rank milega? method='min' kehta hai ke dono 
# ko upar wala chota rank (yani dono ko Rank 1) de do, aur Rank 2 ko skip kar ke agli branch ko seedha Rank 
# 3 do. Is se data mein be-insaafi nahi hoti.


# Pandas jab bhi automatic ranking karta hai, toh woh numbers ko points (floats) mein nikaalta hai, jaise 1.0, 2.0, 3.0.
#     to hum ny int mn con kr diya

branch_scorecard['Rank'] = branch_scorecard['Total_Revenue'].rank(ascending=False, method='min').astype(int)

# 3. Table ko Rank ke mutabiq sort kar ke print karna
branch_scorecard = branch_scorecard.sort_values('Rank')
print("--- BRANCH PERFORMANCE SCORECARD ---")

# .to_string(): Pandas mein jab hum kisi table ko print karte hain, toh agar table badi ho toh Pandas beech
# ke columns ya rows ko chupa deta hai (dots ... dikha deta hai). .to_string() lagane se Pandas poori table 
# ko ek lambi string (text) mein badal deta hai, jisse poori table bina kuch chupe screen par nazar aati hai.

# : Pandas har table ke shuru mein automatic ek default numbering laga deta hai (0, 1, 2...). 
#     Kyunki humne apni table mein Rank (1, 2, 3) ka column khud bana liya hai, isliye humein woh 
# faltu wali numbering (0, 1, 2) nahi chahiye thi. index=False likhne se woh extra numbers chup jate hain.
print(branch_scorecard.to_string(index=False))

--- BRANCH PERFORMANCE SCORECARD ---
Branch  Total_Revenue  Avg_Transaction_Value  Avg_Rating  Num_Transactions Most_Popular_Product  Rank
  Giza    110568.7065             337.099715    7.072866               328   Food and beverages     1
  Alex    106200.3705             312.354031    7.027059               340   Home and lifestyle     2
 Cairo    106197.6720             319.872506    6.818072               332  Fashion accessories     3


In [ ]:
# Branch Performance Summary

# Branch Giza leads in total revenue (110,569) and has the highest average customer rating (7.07), 
# making it the strongest performing location overall. Branch Alex has the most transactions (340) 
# but lower revenue per transaction, suggesting customers are buying cheaper items — this could be an upselling opportunity.
# I recommend the owner investigate why Cairo has the lowest average rating (6.82) as improving customer experience there may
# have the highest return on investment.